You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [1]:
import pandas as pd
import numpy as np 

In [2]:
# read data 
bank_marketing_df = pd.read_csv("bank_marketing.csv")
# display data 
bank_marketing_df.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no


In [3]:
# make a copy of data 
df = bank_marketing_df.copy()

# take a look at the data 
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_id                   41188 non-null  int64  
 1   age                         41188 non-null  int64  
 2   job                         41188 non-null  str    
 3   marital                     41188 non-null  str    
 4   education                   41188 non-null  str    
 5   credit_default              41188 non-null  str    
 6   mortgage                    41188 non-null  str    
 7   month                       41188 non-null  str    
 8   day                         41188 non-null  int64  
 9   contact_duration            41188 non-null  int64  
 10  number_contacts             41188 non-null  int64  
 11  previous_campaign_contacts  41188 non-null  int64  
 12  previous_outcome            41188 non-null  str    
 13  cons_price_idx              41188 non-null

In [4]:
# cleaning process
"""
1 - in job column replace '.' with '_'
2 - in education column replace '.' with '_' and "unknow" with np.Nan
3 - in credit_default and mortgage and campaign_outcome columns convert to boolen datatype 1 if 'yes' else 0
4 - in previous_outcome column convert to boolen datatype 1 if 'success' else 0
5 - Create from a combination of day, month, and a newly created year column (which should have a value of 2022);
Format = "YYYY-MM-DD"

"""
# task 1
df['job'] = df['job'].str.replace(".", "_")

# task 2
df['education'] = df['education'].str.replace(".", "_")
df['education'] = df['education'].apply(lambda x : np.nan if x == 'unknown' else x)

# task 3
df['credit_default'] = df['credit_default'].apply(lambda x: 1 if x=='yes' else 0)
df['mortgage'] = df['mortgage'].apply(lambda x: 1 if x=='yes' else 0)
df['campaign_outcome'] = df['campaign_outcome'].apply(lambda x: 1 if x=='yes' else 0)


# task 4
df['previous_outcome'] = df['previous_outcome'].apply(lambda x: 1 if x =='success' else 0)


# date creation task 
# first add year column in the data
df['year'] = "2022"

# convert day datatye to str 
df['day'] = df['day'].astype(str)

# Add last_contact_date column
df['last_contact_date'] = df['year'] + "-" + df['month'] + "-" + df['day']

df['last_contact_date'] = pd.to_datetime(df['last_contact_date'], format= "%Y-%b-%d")
# df['last_contact_date']



In [5]:
df['education'].unique()

<StringArray>
[           'basic_4y',         'high_school',            'basic_6y',
            'basic_9y', 'professional_course',                   nan,
   'university_degree',          'illiterate']
Length: 8, dtype: str

In [6]:
# now spliting the data 

client = df[['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage']]
campaign = df[['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'last_contact_date']]
economics = df[['client_id', 'cons_price_idx', 'euribor_three_months']]



# Clean and convert outcome columns to bool
for col in ["campaign_outcome", "previous_outcome"]:
  campaign[col] = campaign[col].astype(bool)

# Clean and convert outcome columns to bool
for col in ['credit_default', 'mortgage']:
  client[col] = client[col].astype(bool)


In [7]:
# Save tables to individual csv files
client.to_csv("client.csv", index=False)
campaign.to_csv("campaign.csv", index=False)
economics.to_csv("economics.csv", index=False)

In [8]:
client['education'].unique()

<StringArray>
[           'basic_4y',         'high_school',            'basic_6y',
            'basic_9y', 'professional_course',                   nan,
   'university_degree',          'illiterate']
Length: 8, dtype: str